In [0]:
# ==========================================
# GOLD LAYER — Bloco 1: Setup
# ==========================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG       = "workspace"
SILVER_SCHEMA = "qap_2025_silver"
GOLD_SCHEMA   = "qap_2025_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# Verifica se a Bridge existe antes de continuar
BRIDGE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.bridge_team_day_context"
if not spark.catalog.tableExists(BRIDGE_TABLE):
    raise ValueError(f"[ERRO] Bridge não encontrada: {BRIDGE_TABLE}. Rode o Silver Bloco 5 primeiro.")

df_bridge = spark.table(BRIDGE_TABLE)

print(f"[OK] Bridge carregada com {df_bridge.count()} linhas.")
print(f"[OK] Colunas disponíveis: {df_bridge.columns}")

In [0]:
# ==========================================
# GOLD LAYER — Bloco 2: Ranking de Equipes
# ==========================================

# Janela para ranking global entre equipes
window_rank = Window.partitionBy(F.lit(1)).orderBy(F.desc("avg_fpy"))

gold_team_ranking = (
    df_bridge.groupBy("team", "shift")
    .agg(
        # Qualidade
        F.round(F.avg("quality_actual_fpy"), 4).alias("avg_fpy"),
        F.round(F.avg("fpy_gap"), 4).alias("avg_fpy_gap"),
        F.round(F.avg("rework_rate"), 4).alias("avg_rework_rate"),
        F.sum("rework_units").alias("total_rework_units"),

        # Manutenção
        F.sum("breakdown_events").alias("total_breakdowns"),
        F.round(F.avg("mttr_minutes_avg"), 2).alias("avg_mttr_minutes"),

        # Engenharia
        F.round(F.avg("cycle_gap_pct"), 4).alias("avg_cycle_gap_pct"),
        F.sum(F.when(F.col("needs_improvement_to_meet_takt") == True, 1).otherwise(0)).alias("days_below_takt"),

        # Contagem
        F.count("date").alias("days_measured"),
        F.max("silver_ts").alias("last_silver_update")
    )
    .withColumn("team_rank_by_fpy", F.rank().over(window_rank))
    .withColumn("gold_ts", F.current_timestamp())
)

gold_team_ranking.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.agg_team_performance_ranking")

print(f"[OK] agg_team_performance_ranking: {gold_team_ranking.count()} linhas")
display(gold_team_ranking.orderBy("team_rank_by_fpy"))

In [0]:
# ==========================================
# GOLD LAYER — Bloco 3: Oportunidades Diárias
# ==========================================

gold_improvement_daily = (
    df_bridge
    .select(
        "date", "team", "shift",
        "quality_status",
        "quality_actual_fpy", "fpy_gap",
        "rework_rate", "rework_units",
        "breakdown_events", "mttr_minutes_avg",
        "cycle_gap_pct", "needs_improvement_to_meet_takt",
        "avg_cycle_time_sec", "takt_time_sec"
    )
    .withColumn("priority_score",
        F.when(
            (F.col("needs_improvement_to_meet_takt") == True) &
            (F.col("quality_status") == "CRITICAL"), F.lit("CRITICAL")
        ).when(
            F.col("needs_improvement_to_meet_takt") == True, F.lit("HIGH_CYCLE")
        ).when(
            F.col("rework_rate") > 0.05, F.lit("HIGH_REWORK")
        ).when(
            F.col("mttr_minutes_avg") > 60, F.lit("MAINTENANCE_DELAY")
        ).when(
            F.col("quality_status") == "WARNING", F.lit("QUALITY_WARNING")
        ).otherwise(F.lit("STABLE"))
    )
    .withColumn("gold_ts", F.current_timestamp())
)

gold_improvement_daily.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.agg_improvement_opportunities_daily")

print(f"[OK] agg_improvement_opportunities_daily: {gold_improvement_daily.count()} linhas")
display(
    gold_improvement_daily
    .filter(F.col("priority_score") != "STABLE")
    .orderBy(F.desc("date"))
)

In [0]:
# ==========================================
# GOLD LAYER — Bloco 4: KPIs Mensais
# ==========================================

df_monthly = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.production_monthly")

gold_monthly_kpis = (
    df_monthly
    .withColumn("production_achievement_pct",
        F.round(F.col("production_actual_units") / F.col("production_target_units"), 4))
    .withColumn("cost_variance",
        F.col("monthly_cost_actual") - F.col("monthly_cost_target"))
    .withColumn("cost_variance_pct",
        F.round(
            (F.col("monthly_cost_actual") - F.col("monthly_cost_target")) / F.col("monthly_cost_target"), 4
        ))
    .withColumn("production_loss_variance",
        F.col("production_loss_actual_minutes") - F.col("production_loss_target_minutes"))
    .withColumn("absenteeism_variance",
        F.col("absenteeism_actual_events") - F.col("absenteeism_target_events"))
    .withColumn("performance_status",
        F.when(F.col("production_achievement_pct") >= 0.98, "ON_TARGET")
         .when(F.col("production_achievement_pct") >= 0.93, "ACCEPTABLE")
         .otherwise("BELOW_TARGET"))
    .withColumn("gold_ts", F.current_timestamp())
)

gold_monthly_kpis.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.agg_production_monthly_kpis")

print(f"[OK] agg_production_monthly_kpis: {gold_monthly_kpis.count()} linhas")
display(gold_monthly_kpis.orderBy("month"))

In [0]:
# ==========================================
# GOLD LAYER — Bloco 5: Headcount por Equipe
# ==========================================

df_hr = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.hr_employees")

# Conta funcionários ativos por equipe (sem data de saída ou saída futura)
gold_headcount = (
    df_hr
    .filter(
        F.col("end_date").isNull() |
        (F.col("end_date").cast("date") >= F.current_date())
    )
    .groupBy("team", "role", "department")
    .agg(
        F.count("employee_id").alias("active_headcount"),
        F.round(F.avg("age"), 1).alias("avg_age"),
        F.sum(F.when(F.col("married") == True, 1).otherwise(0)).alias("married_count")
    )
    .withColumn("gold_ts", F.current_timestamp())
)

gold_headcount.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_team_headcount")

print(f"[OK] dim_team_headcount: {gold_headcount.count()} linhas")
display(gold_headcount.orderBy("team", "role"))

In [0]:
# ==========================================
# GOLD LAYER — Bloco 6: Resumo
# ==========================================

tabelas = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}")

print("=" * 55)
print("GOLD LAYER — TABELAS CRIADAS")
print("=" * 55)
for row in tabelas.collect():
    nome = f"{CATALOG}.{GOLD_SCHEMA}.{row['tableName']}"
    try:
        qtd = spark.table(nome).count()
        print(f"  {row['tableName']:<40} {qtd:>5} linhas")
    except:
        print(f"  {row['tableName']:<40} [erro ao contar]")
print("=" * 55)
print("""
TABELAS DISPONÍVEIS PARA DASHBOARDS:
  → agg_team_performance_ranking       (Ranking por equipe/turno)
  → agg_improvement_opportunities_daily (Desvios diários por prioridade)
  → agg_production_monthly_kpis        (KPIs mensais de produção)
  → dim_team_headcount                 (Headcount ativo por equipe)
""")